<a href="https://colab.research.google.com/github/SanjaySaatyaki/llm-from-scratch/blob/main/self-attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Simple Attention

In [1]:
import torch
inputs = torch.tensor(
    [[0.43,0.15,0.89],
     [0.55,0.87,0.66],
     [0.57,0.85,0.64],
     [0.22,0.58,0.33],
     [0.77,0.25,0.10],
     [0.05,0.80,0.55]]

)

In [2]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
attn_scores_2

tensor([0., 0., 0., 0., 0., 0.])

### What is a Dot Product?

A dot product (also known as the scalar product) is an algebraic operation that takes two equal-length sequences of numbers (usually coordinate vectors) and returns a single number.

In the context of the code provided earlier (`torch.dot(x_i, query)`), the dot product is used to calculate a measure of similarity between two vectors, `x_i` (one of the input vectors) and `query` (the query vector).

**Mathematically, for two vectors A and B:**

If $A = [a_1, a_2, ..., a_n]$ and $B = [b_1, b_2, ..., b_n]$, then the dot product is calculated as:

$A \cdot B = \sum_{i=1}^{n} a_i b_i = a_1 b_1 + a_2 b_2 + ... + a_n b_n$

**Example from your notebook:**

For `x_i = [0.43, 0.15, 0.89]` and `query = [0.55, 0.87, 0.66]` (from `inputs[1]`):

`0.43 * 0.55 + 0.15 * 0.87 + 0.89 * 0.66 = 0.2365 + 0.1305 + 0.5874 = 0.9544`

This matches the first value in your `attn_scores_2` tensor, which you manually verified in cell `HtHC1lY96lVC`.

**In the context of Attention Mechanisms:**

The dot product is frequently used to calculate attention scores, where it measures how much a 'query' (e.g., the current word in a sentence) 'attends' to different 'keys' (e.g., other words in the sentence). A higher dot product typically means higher similarity and thus higher attention.

In [3]:
for i, x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [4]:
def softmax_naive(x):
  return torch.exp(x) / torch.exp(x).sum(dim=0)

In [5]:
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [6]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

context vector

In [7]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
  context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


## Genralize for all steps

In [8]:
attn_scores = torch.empty(6,6)

In [9]:
for i, x_i in enumerate(inputs):
  for j, x_j in enumerate(inputs):
    attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [10]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [11]:
attn_weights = torch.softmax(attn_scores, dim=-1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [12]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


# Self Attention

In [13]:
x_2 = inputs[i]
d_in = inputs.shape[1]
d_out = 2